In [ ]:
import pandas as pd, numpy as np

df = pd.read_csv("digital_marketing_dataset_dirty.csv")
print(df.shape)

(30300, 35)


In [ ]:
df = df.drop_duplicates()
df = df.drop(columns=["is_holiday (text)", "is_weekend (text)"])

for c in df.select_dtypes(include=["object", "string"]).columns:
    df[c] = df[c].astype("string").str.strip()

In [ ]:
cmap = {"ksa":"KSA","uae":"United Arab Emirates","usa":"United States",
        "united states":"United States","united kingdom":"United Kingdom"}
df["country"] = df["country"].str.lower().map(lambda x: cmap.get(x, x.title()) if pd.notna(x) else x)

for c in ["market_tier","account_type","funnel_stage","objective"]:
    df[c] = df[c].str.title()

df["platform"] = df["platform"].str.lower().replace({
    "instagram":"Meta","facebook":"Meta","meta":"Meta",
    "tik tok":"TikTok","tiktok":"TikTok","google search":"Google Search",
    "google display":"Google Display","linkedin":"LinkedIn","snapchat":"Snapchat"})

In [ ]:
EGP_TO_USD = 1/50   # assumption: change to the rate you want

def to_num(s):
    s = s.astype("string")
    is_egp = s.str.contains("EGP", na=False)
    n = pd.to_numeric(s.str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce")
    return n.where(~is_egp, n * EGP_TO_USD)

for c in ["spend","impressions","clicks","revenue"]:
    df[c] = to_num(df[c])

In [ ]:
def parse_date(s):
    out = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    for f in ["%d/%m/%Y", "%m/%d/%Y"]:
        out = out.fillna(pd.to_datetime(s, format=f, errors="coerce"))
    return out

df["date"] = parse_date(df["date"])
df = df.dropna(subset=["date"])          # drops ~363 rows with no date

# rebuild date parts from the clean date so month/year/week can't be wrong
df["year"] = df.date.dt.year
df["month"] = df.date.dt.month
df["month_name"] = df.date.dt.strftime("%b")
df["week"] = df.date.dt.isocalendar().week.astype(int)
df["day_of_week"] = df.date.dt.strftime("%a")
df["is_weekend"] = (df.date.dt.dayofweek >= 5).astype(int)

In [ ]:
big = 9_999_999
for c in ["spend","impressions","clicks","revenue","conversions","reach","video_views"]:
    df.loc[(df[c] < 0) | (df[c] >= big), c] = np.nan

df.loc[df.reach >= 1e8, "reach"] = np.nan
df.loc[df.conversions >= 1e5, "conversions"] = np.nan
df.loc[~df.post_hour.between(0, 23), "post_hour"] = np.nan
df.loc[(df.frequency <= 0) | (df.frequency >= 100), "frequency"] = np.nan

for c in ["country","platform","account","campaign_name"]:
    df[c] = df[c].fillna("Unknown")

In [ ]:
df.loc[df.reach > df.impressions, "reach"] = np.nan
df["frequency"] = (df.impressions / df.reach).round(2)
df.to_csv("digital_marketing_clean.csv", index=False)

In [ ]:
print(df.shape)
print(df.isna().sum()[lambda x: x > 0])
print(df[["spend","impressions","clicks","revenue"]].describe())
print(df.country.value_counts())
print(df.platform.value_counts())

df.to_csv("digital_marketing_clean.csv", index=False)
files.download("digital_marketing_clean.csv")

(29775, 33)
post_hour       59
spend          553
impressions    484
reach          152
frequency      635
clicks         475
conversions    481
revenue        550
dtype: int64
            spend   impressions      clicks      revenue
count     29222.0       29291.0     29300.0      29225.0
mean   420.978791    56384.1876  852.990171   476.094045
std    463.253763  80132.571675   1483.3349  2810.246001
min          15.0         693.0         3.0          0.0
25%       85.5825        9304.0       108.0          0.0
50%        229.64       25133.0       311.0          0.0
75%      608.6275       69014.0       965.0        73.67
max       3759.94     1015972.0     30884.0    104733.41
country
KSA                     5359
Iraq                    4526
United Arab Emirates    3438
Egypt                   2944
Morocco                 2885
Lebanon                 1760
Yemen                   1518
Kuwait                  1489
Qatar                   1424
Oman                    1408
Bahrain     

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>